<center><h1>Soundalgekar_Abhishek_Project_VGG16</h1></center>

Name: Abhishek Soundalgekar
<br>
Github Username: ABHISHEK SOUNDALGEKAR
<br>
USC ID: 2089011000

In [8]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from sklearn.metrics import classification_report
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

In [7]:
DATA_ROOT   = "../data/WasteSplit"
TRAIN_PATH  = os.path.join(DATA_ROOT, "train")
VALID_PATH  = os.path.join(DATA_ROOT, "val")
TEST_PATH   = os.path.join(DATA_ROOT, "test")

IMG_SIZE    = (224, 224)
BATCH_SZ    = 5
MAX_EPOCHS  = 100
LR          = 1e-4
PATIENCE    = 40
L2_FACTOR   = 0.01
DROPOUT_RATE = 0.2

# --- Data Augmentation & Generators ---
augmenter = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.vgg16.preprocess_input,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    brightness_range=(0.8, 1.2),
    horizontal_flip=True
)

train_loader = augmenter.flow_from_directory(
    TRAIN_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SZ,
    class_mode='categorical',
    shuffle=True
)
valid_loader = augmenter.flow_from_directory(
    VALID_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SZ,
    class_mode='categorical',
    shuffle=False
)
test_loader = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.vgg16.preprocess_input
).flow_from_directory(
    TEST_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SZ,
    class_mode='categorical',
    shuffle=False
)

#Load VGG16 backbone
backbone = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=IMG_SIZE + (3,)
)
backbone.trainable = False

#new classification head
x = backbone.output
x = GlobalAveragePooling2D()(x)
# Dense block 1
x = Dense(256, activation='relu', kernel_regularizer=l2(L2_FACTOR))(x)
x = BatchNormalization()(x)
# Dense block 2
x = Dense(128, activation='relu', kernel_regularizer=l2(L2_FACTOR))(x)
x = BatchNormalization()(x)
x = Dropout(DROPOUT_RATE)(x)
# Dense block 3
x = Dense(64, activation='relu', kernel_regularizer=l2(L2_FACTOR))(x)
x = BatchNormalization()(x)

#softmax layer for all classes
num_classes = train_loader.num_classes
predictions = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=backbone.input, outputs=predictions)

model.compile(
    optimizer=Adam(learning_rate=LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stopper = EarlyStopping(
    monitor='val_loss',
    patience=PATIENCE,
    restore_best_weights=True
)

history = model.fit(
    train_loader,
    validation_data=valid_loader,
    epochs=MAX_EPOCHS,
    callbacks=[early_stopper]
)

test_loss, test_acc = model.evaluate(test_loader)
print(f"Test Accuracy: {test_acc:.4f}")


Found 3323 images belonging to 9 classes.
Found 710 images belonging to 9 classes.
Found 719 images belonging to 9 classes.
Epoch 1/100
665/665 ━━━━━━━━━━━━━━━━━━━━ 1044s 2s/step - accuracy: 0.1610 - loss: 8.4483 - val_accuracy: 0.3437 - val_loss: 7.7222
Epoch 2/100
665/665 ━━━━━━━━━━━━━━━━━━━━ 1018s 2s/step - accuracy: 0.3032 - loss: 7.7022 - val_accuracy: 0.4577 - val_loss: 7.1322
Epoch 3/100
665/665 ━━━━━━━━━━━━━━━━━━━━ 991s 1s/step - accuracy: 0.3788 - loss: 7.2778 - val_accuracy: 0.5479 - val_loss: 6.7923
Epoch 4/100
665/665 ━━━━━━━━━━━━━━━━━━━━ 1000s 2s/step - accuracy: 0.4618 - loss: 6.9138 - val_accuracy: 0.5901 - val_loss: 6.6140
Epoch 5/100
665/665 ━━━━━━━━━━━━━━━━━━━━ 1015s 2s/step - accuracy: 0.4920 - loss: 6.6268 - val_accuracy: 0.6000 - val_loss: 6.3040
Epoch 6/100
665/665 ━━━━━━━━━━━━━━━━━━━━ 1012s 2s/step - accuracy: 0.5193 - loss: 6.4082 - val_accuracy: 0.6225 - val_loss: 6.1063
Epoch 7/100
665/665 ━━━━━━━━━━━━━━━━━━━━ 1000s 2s/step - accuracy: 0.5381 - loss: 6.2041 - 


KeyboardInterrupt

